> **Research record.** Cleaned copy of `800-research-users-locations/1006-place2vec-ngrams-third-places.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Visit-sequence n-grams by home neighbourhood, with Google Places third-place flags.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# Scheme 3 n-grams with Google Places status flags

Downstream of [`01-embeddings-3periods.ipynb`](01-embeddings-3periods.ipynb) **STEP 5** (Scheme 3, split by home neighborhood).

For each period and home neighborhood type, extract top trigrams. Every POI token is tagged with **exactly one** status:

| Status | Rule |
|--------|------|
| `third_place` | PU `prediction_third_place = 1` |
| `potential_but_not_third_place` | in `google_pois_features`, `flag_location_suits_rw = 1`, not detected |
| `likely_office` | in `google_pois_features`, `flag_location_suits_rw = 0` |
| `non_google` | POI id not in `google_pois_features` |

Category: Google `primarytype` when the id is in features; otherwise keep merged `pois_category` (e.g. OSM `food`).

Example tokens:

- `POI_coffee_shop_third_place_WEEKDAY_LATE_MORNING`
- `POI_food_non_google_WEEKDAY_LATE_MORNING`

**Prerequisites:** Scheme 3 corpora on GCS from 1003 (`corpus_{period}_scheme3.txt`). Section 5 also needs `models/place2vec_post_1year_scheme3.model`.

Section 5 plots Google Places only: cosine similarity to the work centroid vs distance to CBD, one scatter per weekday day-part.

## Section 0 — Setup

In [ ]:
import importlib
import sys
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

import place2vec
from geofunctions import utils

importlib.reload(place2vec)

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

GS_BUCKET = "phd_remote_work_data"
GS_PREFIX = "place2vec"
SCHEME = 3
PERIODS = list(place2vec.PERIODS.keys())  # pre_covid, post_immediate, post_1year

FEATURES_TABLE = "phd-habidatum.phd_pois.google_pois_features"
PU_TABLE = "phd-habidatum.phd_models_results.google_3rd_places_detected_pu_20260601"

HOME_NEIGHBORHOODS = [
    "satellite_city",
    "residential_remote",
    "residential_central",
    "CBD",
]

STATUS_THIRD = "third_place"
STATUS_POTENTIAL = "potential_but_not_third_place"
STATUS_OFFICE = "likely_office"
STATUS_NON_GOOGLE = "non_google"

TOP_K = 15
NGRAM_N = 3

## Section 1 — Build POI status + category maps

Load Google features (`flag_location_suits_rw`, `primarytype`) and PU detections. Print status counts and category×status dicts for sanity.

In [ ]:
df_features = utils.read_query_to_dataframe(
    f"""
    SELECT
      CAST(id AS STRING) AS id,
      primarytype,
      flag_location_suits_rw
    FROM `{FEATURES_TABLE}`
    """
)

df_detected = utils.read_query_to_dataframe(
    f"""
    SELECT CAST(id AS STRING) AS id
    FROM `{PU_TABLE}`
    WHERE prediction_third_place = 1
    """
)

detected_ids = set(df_detected["id"].astype(str))
features_ids = set(df_features["id"].astype(str))

# Google primarytype for category override (OSM pois_category remains fallback in place2vec)
poi_category_by_id = {}
for _, row in df_features.iterrows():
    pid = str(row["id"])
    pt = row.get("primarytype")
    if pd.notna(pt) and str(pt).strip():
        poi_category_by_id[pid] = str(pt).strip()

# Four-way status for every Google features id; missing ids → non_google at extract time
poi_status_by_id = {}
for _, row in df_features.iterrows():
    pid = str(row["id"])
    if pid in detected_ids:
        poi_status_by_id[pid] = STATUS_THIRD
    elif int(row["flag_location_suits_rw"] or 0) == 1:
        poi_status_by_id[pid] = STATUS_POTENTIAL
    else:
        poi_status_by_id[pid] = STATUS_OFFICE

# Detected ids not in features still count as third_place
for pid in detected_ids:
    poi_status_by_id.setdefault(pid, STATUS_THIRD)

status_counts = Counter(poi_status_by_id.values())
status_counts_dict = {
    STATUS_THIRD: status_counts.get(STATUS_THIRD, 0),
    STATUS_POTENTIAL: status_counts.get(STATUS_POTENTIAL, 0),
    STATUS_OFFICE: status_counts.get(STATUS_OFFICE, 0),
    STATUS_NON_GOOGLE: "(assigned at extract for POIs not in google_pois_features)",
}

print("Status counts (Google features + detected):")
display(status_counts_dict)
print(f"Google features ids: {len(features_ids):,}")
print(f"Detected (PU) ids: {len(detected_ids):,}")
print(f"Category overrides (primarytype): {len(poi_category_by_id):,}")
print(f"Detected not in features: {len(detected_ids - features_ids):,}")

# category × status among Google features
cat_status = defaultdict(Counter)
for pid, status in poi_status_by_id.items():
    cat = poi_category_by_id.get(pid, "UNKNOWN_PRIMARYTYPE")
    cat_status[cat][status] += 1

cat_status_dict = {
    cat: dict(counts)
    for cat, counts in sorted(cat_status.items(), key=lambda x: -sum(x[1].values()))
}
print("\nTop primarytype × status (up to 25 categories):")
display(dict(list(cat_status_dict.items())[:25]))

# Sample id → (category, status)
sample_rows = []
for status in (STATUS_THIRD, STATUS_POTENTIAL, STATUS_OFFICE):
    for pid, st in poi_status_by_id.items():
        if st == status:
            sample_rows.append(
                {
                    "poi_id": pid,
                    "category": poi_category_by_id.get(pid, ""),
                    "status": st,
                }
            )
            break
print("\nSample poi_id → (category, status):")
display(pd.DataFrame(sample_rows))

## Section 2 — Extract Scheme 3 top n-grams by home neighborhood

Uses `aggregate_by_category=True` and `split_by_home_neighborhood=True` so POI tokens become
`POI_{category}_{status}_WEEKDAY_{period}`.

In [ ]:
top_ngrams_dict = {}

for period_name in PERIODS:
    corpus_uri = f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
    print(f"\n=== {period_name} — {corpus_uri} ===")
    line_corpus = place2vec.GCSLineSentences(corpus_uri)
    top_ngrams = place2vec.extract_top_ngrams(
        line_corpus,
        n=NGRAM_N,
        top_k=TOP_K,
        aggregate_by_category=True,
        aggregate_by_neighborhood_type=False,
        aggregate_by_context=False,
        split_by_home_neighborhood=True,
        poi_status_by_id=poi_status_by_id,
        poi_category_by_id=poi_category_by_id,
    )
    top_ngrams_dict[period_name] = top_ngrams
    print(f"Home types: {sorted(top_ngrams.keys())}")
    for home_type, items in sorted(top_ngrams.items()):
        print(f"  {home_type}: {len(items)} n-grams")

## Section 3 — Plot top n-grams per neighborhood × period

Bar color priority (strongest wins):

1. `third_place` → purple
2. `potential_but_not_third_place` → orange
3. `likely_office` → brown
4. `non_google` → teal
5. Else 1003 trajectory logic (work/edu blue, morning-RW red, afternoon gray, evening green)

In [ ]:
COLOR_THIRD = "#7b2cbf"  # purple
COLOR_POTENTIAL = "#e67e22"  # orange
COLOR_OFFICE = "#8c564b"  # brown
COLOR_NON_GOOGLE = "#17becf"  # teal
COLOR_WORK_EDU = "#1f77b4"  # blue
COLOR_RW_MORNING = "#d62728"  # red
COLOR_AFTERNOON = "#b0b0b0"  # gray
COLOR_EVENING = "#2ca02c"  # green

COLOR_LEGEND = {
    COLOR_THIRD: "Third place (detected)",
    COLOR_POTENTIAL: "Potential but not third place",
    COLOR_OFFICE: "Likely office (flag=0)",
    COLOR_NON_GOOGLE: "Non-Google POI",
    COLOR_WORK_EDU: "Work / education trajectory",
    COLOR_RW_MORNING: "Morning RW / home-only",
    COLOR_AFTERNOON: "Afternoon (no work/edu)",
    COLOR_EVENING: "Evening routine",
}

# Longer status strings first so potential_but_not_third_place is not matched as third_place
STATUS_LABELS = [
    (STATUS_POTENTIAL, "[potential not TP]"),
    (STATUS_THIRD, "[third place]"),
    (STATUS_OFFICE, "[likely office]"),
    (STATUS_NON_GOOGLE, "[non-google]"),
]


def _token_status(token):
    """Return the Google status segment in token, if any (exclusive).

    Check potential_but_not_third_place before third_place because the former
    contains the latter as a substring.
    """
    if STATUS_POTENTIAL in token:
        return STATUS_POTENTIAL
    if STATUS_THIRD in token:
        return STATUS_THIRD
    if STATUS_OFFICE in token:
        return STATUS_OFFICE
    if STATUS_NON_GOOGLE in token:
        return STATUS_NON_GOOGLE
    return None


def _ngram_has_status(pattern_tuple, status):
    return any(_token_status(t) == status for t in pattern_tuple)


def _ngram_bar_color(pattern_tuple):
    """Assign bar color; Google status flags take priority over trajectory colors."""
    statuses = {_token_status(t) for t in pattern_tuple}
    if STATUS_POTENTIAL in statuses:
        return COLOR_POTENTIAL
    if STATUS_THIRD in statuses:
        return COLOR_THIRD
    if STATUS_OFFICE in statuses:
        return COLOR_OFFICE
    if STATUS_NON_GOOGLE in statuses:
        return COLOR_NON_GOOGLE

    has_work_or_edu = any("WORK" in t or "POI_education" in t for t in pattern_tuple)
    rw_activity = (
        any("_MORNING" in t for t in pattern_tuple) and not has_work_or_edu
    ) or sum("HOME" in t for t in pattern_tuple) == 3
    unidentified_activity = (
        any("_AFTERNOON" in t for t in pattern_tuple) and not has_work_or_edu
    )

    if has_work_or_edu:
        return COLOR_WORK_EDU
    if rw_activity:
        return COLOR_RW_MORNING
    if unidentified_activity:
        return COLOR_AFTERNOON
    return COLOR_EVENING


def _clean_clean_ngram_label(pattern_tuple):
    label = " -> ".join(pattern_tuple)
    label = label.replace("POI_", "").replace("HOME", "Home").replace("WORK", "Work")
    label = label.replace("_WEEKDAY", "")
    for status, pretty in STATUS_LABELS:
        label = label.replace(status, pretty)
    label = label.replace("_", " ").title()
    label = (
        label.replace("[Third Place]", "[third place]")
        .replace("[Potential Not Tp]", "[potential not TP]")
        .replace("[Likely Office]", "[likely office]")
        .replace("[Non-Google]", "[non-google]")
    )
    return label


def plot_top_ngrams(top_ngrams_dict, periods, target_area, figsize=(12, 18)):
    """Three stacked panels (one per period) of top n-grams for one home type."""
    fig, axes = plt.subplots(len(periods), 1, figsize=figsize, sharex=False)
    if len(periods) == 1:
        axes = [axes]
    plt.subplots_adjust(hspace=0.45)

    area_title = target_area.replace("_", " ").title()
    fig.suptitle(
        f"Top Scheme 3 trajectories — home in {area_title}",
        fontsize=16,
        fontweight="bold",
        y=0.995,
    )

    for i, period in enumerate(periods):
        ax = axes[i]
        if period not in top_ngrams_dict or target_area not in top_ngrams_dict[period]:
            ax.set_title(f"{period}: no data")
            ax.axis("off")
            continue

        items = top_ngrams_dict[period][target_area]
        if not items:
            ax.set_title(f"{period}: empty")
            ax.axis("off")
            continue

        patterns = [_clean_clean_ngram_label(pt) for pt, _ in items]
        probs = [prob for _, prob in items]
        bar_colors = [_ngram_bar_color(pt) for pt, _ in items]

        df_plot = pd.DataFrame({"Pattern": patterns, "Probability": probs})
        sns.barplot(
            data=df_plot,
            x="Probability",
            y="Pattern",
            ax=ax,
            palette=bar_colors,
            edgecolor="black",
            errorbar=None,
        )

        ax.set_title(period.replace("_", " ").title(), fontsize=14, fontweight="bold")
        ax.set_xlabel("Probability of trajectory (share of home-type sentences)")
        ax.set_ylabel("")

        for container in ax.containers:
            labels = [f"{bar.get_width() * 100:.2f}%" for bar in container]
            ax.bar_label(container, labels=labels, padding=3, fontsize=9)

        sns.despine(ax=ax)

    legend_handles = [
        Patch(facecolor=c, edgecolor="black", label=lab)
        for c, lab in COLOR_LEGEND.items()
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=4,
        frameon=True,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.02),
    )
    plt.tight_layout()
    plt.show()

    print(
        f"**Fig.** Top Scheme 3 trigrams for residents of {area_title}, "
        f"pre-COVID / post-immediate / post-1year. Bars colored by Google Places "
        f"status when present (third place / potential-not-TP / likely office / "
        f"non-google); otherwise by work/edu vs morning-RW vs afternoon vs evening. "
        f"*Notes:* Corpus scheme 3; Google POIs use `primarytype`; non-Google keep "
        f"`pois_category` (e.g. food); detected = PU `prediction_third_place=1`."
    )


In [ ]:
# Prefer canonical order from config; append any extra home types found in the extract.
found_homes = set()
for period_dict in top_ngrams_dict.values():
    found_homes.update(period_dict.keys())

ordered_homes = [h for h in HOME_NEIGHBORHOODS if h in found_homes]
ordered_homes += sorted(found_homes - set(HOME_NEIGHBORHOODS))
print("Plotting home types:", ordered_homes)

for target_area in ordered_homes:
    print("\n" + "=" * 80)
    print(f"Home neighborhood: {target_area}")
    print("=" * 80)
    plot_top_ngrams(
        top_ngrams_dict=top_ngrams_dict,
        periods=PERIODS,
        target_area=target_area,
    )

## Section 4 — Sanity: status prevalence in top n-grams

In [ ]:
STATUS_KEYS = [STATUS_POTENTIAL, STATUS_THIRD, STATUS_OFFICE, STATUS_NON_GOOGLE]

rows = []
example_labels = []

for period in PERIODS:
    for home_type, items in sorted(top_ngrams_dict.get(period, {}).items()):
        row = {"period": period, "home_type": home_type, "n_top": len(items)}
        for status in STATUS_KEYS:
            row[f"n_with_{status}"] = sum(
                _ngram_has_status(ngram, status) for ngram, _ in items
            )
        rows.append(row)
        for ngram, val in items:
            if any(_ngram_has_status(ngram, s) for s in STATUS_KEYS):
                example_labels.append(
                    {
                        "period": period,
                        "home_type": home_type,
                        "label": _clean_clean_ngram_label(ngram),
                        "raw": " | ".join(ngram),
                        "probability": val,
                    }
                )

df_sanity = pd.DataFrame(rows)

## Section 5 — Cosine similarity to work vs distance to CBD (Google Places, by day-part)

Post-1-year scheme 3. Each panel is one weekday day-part. **Google Places only** (ids in `google_3rdplaces_cities` / `ChIJ*`).

| | |
|---|---|
| **X** | Distance to CBD (km), Azrieli Center |
| **Y** | Cosine(`POI_{id}_WEEKDAY_{period}`, work centroid) |
| **Color** | Google `primarytype` (top 12; rest `other`) |

Work centroid is the mean of all `WORK_*` tokens (shared across panels). A POI appears in a panel only if that day-part token is in the model.

This is the POI-level analogue of 1003 “Save cos similarity to work to BQ”, without the 20M-row POI × work-neighborhood dump.

In [ ]:
import re

import numpy as np
from gensim.models import Word2Vec
from google.cloud import bigquery
from IPython.display import Markdown, display

PERIOD_POST = "post_1year"
MODEL_PATH = f"models/place2vec_{PERIOD_POST}_scheme{SCHEME}.model"
GOOGLE_POIS_TABLE = place2vec.TABLE_GOOGLE_POIS  # phd-habidatum.phd_pois.google_3rdplaces_cities
DAY_PERIODS = list(place2vec.POI_DAY_PERIODS)  # MORNING, LATE_MORNING, AFTERNOON, EVENING
PERIOD_LABELS = {
    "MORNING": "Morning",
    "LATE_MORNING": "Late morning",
    "AFTERNOON": "Afternoon",
    "EVENING": "Evening",
}
POI_RE = re.compile(
    r"^POI_(.+)_WEEKDAY_(MORNING|LATE_MORNING|AFTERNOON|EVENING|NIGHT)$"
)

CBD_LON = 34.79188915373317  # Azrieli Center
CBD_LAT = 32.07472355010219

model = Word2Vec.load(MODEL_PATH, mmap="r")
wv = model.wv

poi_period_by_id = {}
n_night = 0
for token in wv.key_to_index:
    m = POI_RE.match(token)
    if not m:
        continue
    poi_id, day_period = m.groups()
    if day_period == "NIGHT":
        n_night += 1
        continue
    poi_period_by_id.setdefault(poi_id, {})[day_period] = token

vocab_poi_ids = set(poi_period_by_id)
google_vocab_ids = {pid for pid in vocab_poi_ids if pid.startswith("ChIJ")}

print(f"Model: {MODEL_PATH}")
print(f"Vocab size: {len(wv.key_to_index):,}")
print(f"Scheme 3 unique POI ids in vocab: {len(vocab_poi_ids):,}")
print(f"of which Google Place IDs (ChIJ*): {len(google_vocab_ids):,}")
print(f"NIGHT tokens in vocab (not plotted): {n_night:,}")
for p in DAY_PERIODS:
    n_tok = sum(p in periods for periods in poi_period_by_id.values())
    n_g = sum(p in poi_period_by_id[pid] for pid in google_vocab_ids)
    print(f"  {p}: {n_tok:,} POI tokens, of which Google: {n_g:,}")

In [ ]:
# Geometry + primarytype: google_pois_features (Gush Dan, cheap), then
# google_3rdplaces_cities for remaining Google vocab ids.

df_feat_geo = utils.read_query_to_dataframe(
    f"""
    SELECT
      CAST(id AS STRING) AS poi_id,
      LOWER(primarytype) AS primarytype,
      ST_DISTANCE(
        geometry,
        ST_GEOGPOINT({CBD_LON}, {CBD_LAT})
      ) / 1000.0 AS dist_to_cbd_km
    FROM `{FEATURES_TABLE}`
    WHERE geometry IS NOT NULL
    """
)

feat_ids = set(df_feat_geo["poi_id"].astype(str))
n_google_in_features = len(google_vocab_ids & feat_ids)
missing_google_ids = sorted(google_vocab_ids - feat_ids)

print(f"Google vocab ids in `{FEATURES_TABLE}`: {n_google_in_features:,}")
print(f"Google vocab ids not in features (lookup `{GOOGLE_POIS_TABLE}`): {len(missing_google_ids):,}")

geo_frames = [
    df_feat_geo[df_feat_geo["poi_id"].astype(str).isin(google_vocab_ids)].copy()
]

if missing_google_ids:
    client = bigquery.Client(project="phd-habidatum")
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("poi_ids", "STRING", missing_google_ids),
        ]
    )
    query_cities = f"""
    SELECT
      CAST(id AS STRING) AS poi_id,
      LOWER(primarytype) AS primarytype,
      ST_DISTANCE(
        geometry,
        ST_GEOGPOINT({CBD_LON}, {CBD_LAT})
      ) / 1000.0 AS dist_to_cbd_km
    FROM `{GOOGLE_POIS_TABLE}`
    WHERE CAST(id AS STRING) IN UNNEST(@poi_ids)
      AND geometry IS NOT NULL
    """
    df_cities_geo = client.query(query_cities, job_config=job_config).to_dataframe()
    print(f"Matched in google_3rdplaces_cities: {len(df_cities_geo):,}")
    geo_frames.append(df_cities_geo)

df_google_geo = (
    pd.concat(geo_frames, ignore_index=True)
    .drop_duplicates(subset=["poi_id"], keep="first")
)
df_google_geo["poi_id"] = df_google_geo["poi_id"].astype(str)
google_with_geo = set(df_google_geo["poi_id"]) & google_vocab_ids
n_no_geo = len(google_vocab_ids - google_with_geo)

print(f"Google places used in scheme 3 (ChIJ* in vocab): {len(google_vocab_ids):,}")
print(f"  with coordinates: {len(google_with_geo):,}")
if n_no_geo:
    print(f"  without coordinates (dropped from scatter): {n_no_geo:,}")

In [ ]:
# Scheme 3 has WORK_{census_id}, not a bare WORK token. Skip the scheme-0 anchor.
work_centroid, work_tokens = place2vec.build_work_centroid(
    model, work_anchor="__NO_BARE_WORK__"
)
print(f"Work centroid from {len(work_tokens):,} WORK_* tokens")

N_TOP_TYPES = 12
df_geo = df_google_geo[df_google_geo["poi_id"].isin(google_with_geo)].copy()
df_geo["primarytype"] = (
    df_geo["primarytype"].fillna("unknown").replace("", "unknown").astype(str)
)
type_counts = df_geo["primarytype"].value_counts()
top_types = list(type_counts.head(N_TOP_TYPES).index)
print("Top primarytype (Google in vocab with coords):")
print(type_counts.head(N_TOP_TYPES).to_string())

df_geo["category"] = np.where(
    df_geo["primarytype"].isin(top_types), df_geo["primarytype"], "other"
)
palette = plt.cm.tab20(np.linspace(0, 1, N_TOP_TYPES))
type_color = {t: palette[i] for i, t in enumerate(top_types)}
type_color["other"] = (0.55, 0.55, 0.55, 1.0)
type_order = top_types + ["other"]

sim_frames = []
for day_period in DAY_PERIODS:
    pids, tokens = [], []
    for poi_id in df_geo["poi_id"]:
        token = poi_period_by_id.get(poi_id, {}).get(day_period)
        if token is not None and token in wv.key_to_index:
            pids.append(poi_id)
            tokens.append(token)
    if not tokens:
        continue
    mat = np.vstack([wv.get_vector(t, norm=True) for t in tokens])
    sim_frames.append(
        pd.DataFrame(
            {
                "poi_id": pids,
                "day_period": day_period,
                "cosine_to_work": mat @ work_centroid,
            }
        )
    )

df_scatter = pd.concat(sim_frames, ignore_index=True).merge(
    df_geo[["poi_id", "dist_to_cbd_km", "primarytype", "category"]],
    on="poi_id",
    how="inner",
)
print("\nRows in scatter (Google × day-part with token + coords):", f"{len(df_scatter):,}")
print(df_scatter.groupby("day_period").size().reindex(DAY_PERIODS).to_string())
print("\nCosine to work by day-part:")
print(
    df_scatter.groupby("day_period")["cosine_to_work"]
    .describe()
    .reindex(DAY_PERIODS)
    .round(3)
    .to_string()
)

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
fig.suptitle(
    "Google Places: cosine similarity to work vs distance to CBD\n"
    "Post-1-year scheme 3, by weekday day-part",
    fontsize=13,
    fontweight="bold",
    y=0.98,
)

for ax, day_period in zip(axes.ravel(), DAY_PERIODS):
    sub = df_scatter[df_scatter["day_period"] == day_period]
    for cat in type_order:
        s = sub[sub["category"] == cat]
        ax.scatter(
            s["dist_to_cbd_km"],
            s["cosine_to_work"],
            c=[type_color[cat]],
            s=18,
            alpha=0.55,
            linewidths=0,
            label=cat.replace("_", " "),
            zorder=2 if cat != "other" else 1,
        )
    ax.set_title(f"{PERIOD_LABELS[day_period]}  (n = {len(sub):,})")
    ax.set_xlabel("Distance to CBD (km)")
    ax.set_ylabel("Cosine similarity to work")
    ax.axhline(0, color="0.85", linewidth=0.8, zorder=0)
    ax.grid(True, alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Google primary type",
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    markerscale=1.4,
)
fig.tight_layout(rect=[0, 0, 0.82, 0.94])
plt.show()

n_pois = df_scatter["poi_id"].nunique()
caption = (
    f"**Fig.** Cosine similarity of Google Places scheme-3 day-part embeddings to the "
    f"work centroid, by distance to Tel Aviv CBD (Azrieli). Each panel is one weekday "
    f"day-part. Colour is Google `primarytype` (top {N_TOP_TYPES} types; remainder pooled "
    f"as other). *Notes:* n = {n_pois:,} Google Places in the post-1-year scheme-3 "
    f"vocabulary with coordinates; a place appears in a panel only if "
    f"`POI_{{id}}_WEEKDAY_{{period}}` is in the model. Work centroid = mean of "
    f"{len(work_tokens):,} `WORK_*` tokens. OSM / Foursquare POIs excluded. "
    f"Distance is BigQuery `ST_DISTANCE` to Azrieli ({CBD_LAT:.4f}°N, {CBD_LON:.4f}°E)."
)
display(Markdown(caption))